# Grupo 3: Cadena de Suministro de Ayuda Humanitaria

## Título y objetivo

Este notebook implementa un modelo reproducible de inventarios y distribución de ayuda humanitaria durante las 72 horas posteriores al terremoto en Ciudad UVG. El modelo distingue centros de acopio, zonas, suministros, rutas y vehículos, y utiliza bloques temporales de 6 horas.

## Imports y configuración

In [1]:
from pathlib import Path
from dataclasses import dataclass
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

HORIZONTE_HORAS = 72
PASO_HORAS = 6
N_PASOS = HORIZONTE_HORAS // PASO_HORAS
N_RUNS = 100
SEMILLA_BASE = 2026

RUTA_EXCEL = Path("Grupo3_CadenaSuministro.xlsx")
HOJA_DATOS = "Datos_Grupo3"

assert N_PASOS == 12
assert N_RUNS >= 30

## Carga del Excel

In [2]:
def cargar_datos_grupo3(
    ruta_excel: str | Path
) -> dict[str, pd.DataFrame]:

    ruta_excel = Path(ruta_excel)

    if not ruta_excel.exists():
        raise FileNotFoundError(
            f"No se encontró el Excel del Grupo 3 en: "
            f"{ruta_excel.resolve()}"
        )

    archivo_excel = pd.ExcelFile(ruta_excel)
    nombres_hojas = archivo_excel.sheet_names

    if HOJA_DATOS not in nombres_hojas:
        raise ValueError(
            f"Falta la hoja requerida '{HOJA_DATOS}'. "
            f"Hojas encontradas: {nombres_hojas}"
        )

    hoja_completa = pd.read_excel(
        archivo_excel,
        sheet_name=HOJA_DATOS,
        header=None,
    )

    encabezados_secciones = {
        "centros": "1. CENTROS DE ACOPIO",
        "consumo": "2. TASAS DE CONSUMO",
        "rutas": "3. RED DE DISTRIBUCIÓN",
        "flota": "4. FLOTA DE VEHÍCULOS",
        "reposicion": "5. REPOSICIÓN DE SUMINISTROS",
        "output_requerido": "OUTPUT REQUERIDO",
    }

    texto_primera_columna = (
        hoja_completa.iloc[:, 0]
        .fillna("")
        .astype(str)
    )

    filas_secciones = {}

    for nombre, texto_buscado in encabezados_secciones.items():
        coincidencias = texto_primera_columna.str.contains(
            texto_buscado,
            case=False,
            regex=False,
        )

        filas_encontradas = hoja_completa.index[
            coincidencias
        ].tolist()

        filas_secciones[nombre] = (
            filas_encontradas[0]
            if filas_encontradas
            else None
        )

    print(f"Archivo cargado: {ruta_excel.name}")
    print(f"Hojas encontradas: {nombres_hojas}")
    print(
        f"Dimensiones de '{HOJA_DATOS}': "
        f"{hoja_completa.shape}"
    )
    print(
        "Filas iniciales de las secciones:",
        filas_secciones,
    )

    return {
        "hoja_completa": hoja_completa.copy(deep=True),
        "filas_secciones": pd.DataFrame(
            list(filas_secciones.items()),
            columns=["seccion", "fila_inicio"],
        ),
    }


datos_crudos = cargar_datos_grupo3(RUTA_EXCEL)

Archivo cargado: Grupo3_CadenaSuministro.xlsx
Hojas encontradas: ['Datos_Grupo3']
Dimensiones de 'Datos_Grupo3': (53, 6)
Filas iniciales de las secciones: {'centros': 4, 'consumo': 11, 'rutas': 17, 'flota': 29, 'reposicion': 35, 'output_requerido': 48}


## Normalización de tablas

### Centro de acopio

In [3]:
def normalizar_centros(
    hoja_completa: pd.DataFrame,
    fila_inicio: int,
) -> pd.DataFrame:

    if fila_inicio is None:
        raise ValueError(
            "No se encontró la sección de centros de acopio."
        )

    fila_encabezados = fila_inicio + 1
    primera_fila_datos = fila_inicio + 2
    ultima_fila_datos = primera_fila_datos + 4

    centros = hoja_completa.iloc[
        primera_fila_datos:ultima_fila_datos,
        0:6,
    ].copy()

    centros.columns = [
        "centro",
        "zona_original",
        "agua_litros",
        "alimentos_raciones",
        "medicamentos_kits",
        "capacidad_almacenamiento_m3",
    ]

    centros["zona"] = centros[
        "zona_original"
    ].str.extract(r"(Z\d+)", expand=False)

    columnas_numericas = [
        "agua_litros",
        "alimentos_raciones",
        "medicamentos_kits",
        "capacidad_almacenamiento_m3",
    ]

    for columna in columnas_numericas:
        centros[columna] = pd.to_numeric(
            centros[columna],
            errors="raise",
        )

    orden_columnas = [
        "centro",
        "zona",
        "zona_original",
        "agua_litros",
        "alimentos_raciones",
        "medicamentos_kits",
        "capacidad_almacenamiento_m3",
    ]

    return centros[orden_columnas].reset_index(drop=True)


fila_centros = int(
    datos_crudos["filas_secciones"]
    .set_index("seccion")
    .loc["centros", "fila_inicio"]
)

df_centros = normalizar_centros(
    datos_crudos["hoja_completa"],
    fila_centros,
)

display(df_centros)

,centro,zona,zona_original,agua_litros,alimentos_raciones,medicamentos_kits,capacidad_almacenamiento_m3
0,Bodega Central CONRED,Z4,Z4 (Este comercial),180000,42000,1200,2400
1,Almacén Norte,Z2,Z2 (Norte residencial),65000,18000,380,900
2,Depósito Sur,Z3,Z3 (Sur industrial),40000,9500,210,550
3,Centro Logístico Z5,Z5,Z5 (Oeste periférico),28000,7200,145,400


### Tasas de consumo

In [4]:
def normalizar_consumo(
    hoja_completa: pd.DataFrame,
    fila_inicio: int,
    paso_horas: int,
) -> pd.DataFrame:

    if fila_inicio is None:
        raise ValueError(
            "No se encontró la sección de tasas de consumo."
        )

    primera_fila_datos = fila_inicio + 2
    ultima_fila_datos = primera_fila_datos + 3

    consumo = hoja_completa.iloc[
        primera_fila_datos:ultima_fila_datos,
        0:4,
    ].copy()

    consumo.columns = [
        "suministro_original",
        "consumo_por_persona_dia",
        "unidad_original",
        "nota",
    ]

    nombres_normalizados = {
        "Agua potable": "agua",
        "Alimentos (raciones)": "alimentos",
        "Medicamentos básicos": "medicamentos",
    }

    consumo["suministro"] = consumo[
        "suministro_original"
    ].map(nombres_normalizados)

    consumo["consumo_por_persona_dia"] = pd.to_numeric(
        consumo["consumo_por_persona_dia"],
        errors="raise",
    )

    consumo["consumo_por_persona_bloque"] = (
        consumo["consumo_por_persona_dia"]
        * paso_horas
        / 24
    )

    unidades_por_bloque = {
        "agua": "litros/persona/bloque",
        "alimentos": "raciones/persona/bloque",
        "medicamentos": "kits/persona/bloque",
    }

    consumo["unidad_por_bloque"] = consumo[
        "suministro"
    ].map(unidades_por_bloque)

    orden_columnas = [
        "suministro",
        "suministro_original",
        "consumo_por_persona_dia",
        "unidad_original",
        "consumo_por_persona_bloque",
        "unidad_por_bloque",
        "nota",
    ]

    return consumo[orden_columnas].reset_index(drop=True)


fila_consumo = int(
    datos_crudos["filas_secciones"]
    .set_index("seccion")
    .loc["consumo", "fila_inicio"]
)

df_consumo = normalizar_consumo(
    datos_crudos["hoja_completa"],
    fila_consumo,
    PASO_HORAS,
)

display(df_consumo)

,suministro,suministro_original,consumo_por_persona_dia,unidad_original,consumo_por_persona_bloque,unidad_por_bloque,nota
0,agua,Agua potable,3.00,litros,0.7500,litros/persona/bloque,Mínimo OPS en emergencia
1,alimentos,Alimentos (raciones),1.00,ración/día,0.2500,raciones/persona/bloque,Ración = 2100 kcal
2,medicamentos,Medicamentos básicos,0.05,kit/persona,0.0125,kits/persona/bloque,1 kit cubre 20 personas/día


### Red de distribución

In [5]:
def normalizar_rutas(
    hoja_completa: pd.DataFrame,
    fila_inicio: int,
) -> pd.DataFrame:

    if fila_inicio is None:
        raise ValueError(
            "No se encontró la sección de la red de distribución."
        )

    primera_fila_datos = fila_inicio + 2
    ultima_fila_datos = primera_fila_datos + 9

    rutas = hoja_completa.iloc[
        primera_fila_datos:ultima_fila_datos,
        0:5,
    ].copy()

    rutas.columns = [
        "ruta_original",
        "distancia_km",
        "tiempo_normal_h",
        "capacidad_vehiculos_viaje",
        "estado_ruta",
    ]

    componentes_ruta = rutas["ruta_original"].str.split(
        r"\s*(?:→|->)\s*",
        n=1,
        expand=True,
        regex=True,
    )

    rutas["origen"] = componentes_ruta[0].str.strip()
    rutas["destino"] = componentes_ruta[1].str.strip()

    columnas_numericas = [
        "distancia_km",
        "tiempo_normal_h",
        "capacidad_vehiculos_viaje",
    ]

    for columna in columnas_numericas:
        rutas[columna] = pd.to_numeric(
            rutas[columna],
            errors="raise",
        )

    rutas["ruta_id"] = [
        f"R{i:02d}"
        for i in range(1, len(rutas) + 1)
    ]

    orden_columnas = [
        "ruta_id",
        "ruta_original",
        "origen",
        "destino",
        "distancia_km",
        "tiempo_normal_h",
        "capacidad_vehiculos_viaje",
        "estado_ruta",
    ]

    return rutas[orden_columnas].reset_index(drop=True)


fila_rutas = int(
    datos_crudos["filas_secciones"]
    .set_index("seccion")
    .loc["rutas", "fila_inicio"]
)

df_rutas = normalizar_rutas(
    datos_crudos["hoja_completa"],
    fila_rutas,
)

display(df_rutas)

,ruta_id,ruta_original,origen,destino,distancia_km,tiempo_normal_h,capacidad_vehiculos_viaje,estado_ruta
0,R01,Bodega Central → Albergue Estadio,Bodega Central,Albergue Estadio,4.2,0.4,8,Disponible
1,R02,Bodega Central → Albergue Z2,Bodega Central,Albergue Z2,7.8,0.9,6,Disponible
2,R03,Bodega Central → Albergue Z5-1,Bodega Central,Albergue Z5-1,12.1,2.8,4,Dañada — velocidad reducida 60%
3,R04,Bodega Central → Albergue Z5-2,Bodega Central,Albergue Z5-2,13.4,3.5,3,Dañada — velocidad reducida 60%
4,R05,Almacén Norte → Albergue Z2,Almacén Norte,Albergue Z2,2.1,0.2,10,Disponible
5,R06,Almacén Norte → Albergue Estadio,Almacén Norte,Albergue Estadio,6.4,0.7,7,Disponible
6,R07,Depósito Sur → Albergue Z3,Depósito Sur,Albergue Z3,3.8,0.4,8,Disponible
7,R08,Centro Log. Z5 → Albergue Z5-1,Centro Log. Z5,Albergue Z5-1,1.2,0.5,6,Parcialmente bloqueada
8,R09,Centro Log. Z5 → Albergue Z5-2,Centro Log. Z5,Albergue Z5-2,2.4,0.9,5,Parcialmente bloqueada


### Flota de vehículos

In [6]:
def normalizar_flota(
    hoja_completa: pd.DataFrame,
    fila_inicio: int,
) -> pd.DataFrame:

    if fila_inicio is None:
        raise ValueError(
            "No se encontró la sección de la flota de vehículos."
        )

    primera_fila_datos = fila_inicio + 2
    ultima_fila_datos = primera_fila_datos + 3

    flota = hoja_completa.iloc[
        primera_fila_datos:ultima_fila_datos,
        0:5,
    ].copy()

    flota.columns = [
        "tipo_vehiculo_original",
        "cantidad_disponible",
        "capacidad_m3",
        "consumo_combustible_gal_km",
        "combustible_disponible_gal",
    ]

    nombres_normalizados = {
        "Camión grande (10 ton)": "camion_grande",
        "Camioneta (2 ton)": "camioneta",
        "Motocicleta (mensajería)": "motocicleta",
    }

    flota["tipo_vehiculo"] = flota[
        "tipo_vehiculo_original"
    ].map(nombres_normalizados)

    columnas_numericas = [
        "cantidad_disponible",
        "capacidad_m3",
        "consumo_combustible_gal_km",
        "combustible_disponible_gal",
    ]

    for columna in columnas_numericas:
        flota[columna] = pd.to_numeric(
            flota[columna],
            errors="raise",
        )

    flota["cantidad_disponible"] = flota[
        "cantidad_disponible"
    ].astype(int)

    orden_columnas = [
        "tipo_vehiculo",
        "tipo_vehiculo_original",
        "cantidad_disponible",
        "capacidad_m3",
        "consumo_combustible_gal_km",
        "combustible_disponible_gal",
    ]

    return flota[orden_columnas].reset_index(drop=True)


fila_flota = int(
    datos_crudos["filas_secciones"]
    .set_index("seccion")
    .loc["flota", "fila_inicio"]
)

df_flota = normalizar_flota(
    datos_crudos["hoja_completa"],
    fila_flota,
)

display(df_flota)

,tipo_vehiculo,tipo_vehiculo_original,cantidad_disponible,capacidad_m3,consumo_combustible_gal_km,combustible_disponible_gal
0,camion_grande,Camión grande (10 ton),8,28.0,0.18,1400
1,camioneta,Camioneta (2 ton),22,8.0,0.09,1800
2,motocicleta,Motocicleta (mensajería),15,0.3,0.03,420


### Reposiciones

In [7]:
def normalizar_reposicion(
    hoja_completa: pd.DataFrame,
    fila_inicio: int,
) -> pd.DataFrame:

    if fila_inicio is None:
        raise ValueError(
            "No se encontró la sección de reposición de suministros."
        )

    primera_fila_datos = fila_inicio + 2

    ultima_fila_datos = primera_fila_datos + 6

    reposicion = hoja_completa.iloc[
        primera_fila_datos:ultima_fila_datos,
        0:5,
    ].copy()

    reposicion.columns = [
        "bloque_original",
        "agua_adicional_litros",
        "alimentos_adicionales_raciones",
        "medicamentos_adicionales_kits",
        "fuente",
    ]

    componentes_tiempo = reposicion[
        "bloque_original"
    ].str.extract(
        r"T(\d+)\s*\((\d+)-(\d+)h\)"
    )

    componentes_tiempo.columns = [
        "bloque",
        "hora_inicio",
        "hora_fin",
    ]

    reposicion[
        ["bloque", "hora_inicio", "hora_fin"]
    ] = componentes_tiempo.astype(int)

    columnas_numericas = [
        "agua_adicional_litros",
        "alimentos_adicionales_raciones",
        "medicamentos_adicionales_kits",
    ]

    for columna in columnas_numericas:
        reposicion[columna] = pd.to_numeric(
            reposicion[columna],
            errors="raise",
        )

    orden_columnas = [
        "bloque",
        "bloque_original",
        "hora_inicio",
        "hora_fin",
        "agua_adicional_litros",
        "alimentos_adicionales_raciones",
        "medicamentos_adicionales_kits",
        "fuente",
    ]

    return reposicion[orden_columnas].reset_index(drop=True)


fila_reposicion = int(
    datos_crudos["filas_secciones"]
    .set_index("seccion")
    .loc["reposicion", "fila_inicio"]
)

df_reposicion = normalizar_reposicion(
    datos_crudos["hoja_completa"],
    fila_reposicion,
)

display(df_reposicion)

,bloque,bloque_original,hora_inicio,hora_fin,agua_adicional_litros,alimentos_adicionales_raciones,medicamentos_adicionales_kits,fuente
0,2,T2 (12-18h),12,18,0,0,0,Sin reposición aún
1,3,T3 (18-24h),18,24,40000,8000,200,Cruz Roja — primer convoy
2,4,T4 (24-30h),24,30,80000,15000,400,Gobierno central
3,6,T6 (36-42h),36,42,120000,22000,600,Ayuda internacional
4,8,T8 (48-54h),48,54,150000,28000,800,Ayuda internacional
5,10,T10 (60-66h),60,66,100000,18000,500,Cruz Roja — segundo convoy


### Output requerido para Grupo 7

In [9]:
def normalizar_output_requerido(
    hoja_completa: pd.DataFrame,
    fila_inicio: int,
) -> pd.DataFrame:

    if fila_inicio is None:
        raise ValueError(
            "No se encontró la sección de output requerido."
        )

    primera_fila_datos = fila_inicio + 2

    output_requerido = (
        hoja_completa.iloc[primera_fila_datos:, 0]
        .dropna()
        .astype(str)
        .str.strip()
    )

    output_requerido = output_requerido[
        output_requerido.ne("")
    ].reset_index(drop=True)

    if len(output_requerido) != 3:
        raise ValueError(
            "Se esperaban 3 outputs requeridos, pero se "
            f"encontraron {len(output_requerido)}."
        )

    nombres_output = [
        "balance_suministros",
        "primer_agotamiento",
        "recomendacion_logistica",
    ]

    tabla_output = pd.DataFrame(
        {
            "output_id": [
                "O01",
                "O02",
                "O03",
            ],
            "nombre_output": nombres_output,
            "descripcion_original": output_requerido,
        }
    )

    return tabla_output


fila_output_requerido = int(
    datos_crudos["filas_secciones"]
    .set_index("seccion")
    .loc["output_requerido", "fila_inicio"]
)

df_output_requerido = normalizar_output_requerido(
    datos_crudos["hoja_completa"],
    fila_output_requerido,
)

display(df_output_requerido)

,output_id,nombre_output,descripcion_original
0,O01,balance_suministros,a) Balance de suministros por bloque de tiempo...
1,O02,primer_agotamiento,b) Identificación del momento en que algún sum...
2,O03,recomendacion_logistica,c) Recomendación de priorización de rutas y fr...


### Función de normalización

In [10]:
def normalizar_datos_grupo3(
    datos: dict[str, pd.DataFrame],
) -> dict[str, pd.DataFrame]:

    claves_requeridas = {
        "hoja_completa",
        "filas_secciones",
    }

    claves_faltantes = claves_requeridas.difference(
        datos.keys()
    )

    if claves_faltantes:
        raise KeyError(
            "Faltan estructuras necesarias para normalizar: "
            f"{sorted(claves_faltantes)}"
        )

    hoja_completa = datos["hoja_completa"]

    filas_secciones = (
        datos["filas_secciones"]
        .set_index("seccion")["fila_inicio"]
        .to_dict()
    )

    secciones_requeridas = [
        "centros",
        "consumo",
        "rutas",
        "flota",
        "reposicion",
        "output_requerido",
    ]

    secciones_no_encontradas = [
        seccion
        for seccion in secciones_requeridas
        if pd.isna(filas_secciones.get(seccion))
    ]

    if secciones_no_encontradas:
        raise ValueError(
            "No se localizaron las siguientes secciones: "
            f"{secciones_no_encontradas}"
        )

    df_centros_normalizado = normalizar_centros(
        hoja_completa,
        int(filas_secciones["centros"]),
    )

    df_consumo_normalizado = normalizar_consumo(
        hoja_completa,
        int(filas_secciones["consumo"]),
        PASO_HORAS,
    )

    df_rutas_normalizado = normalizar_rutas(
        hoja_completa,
        int(filas_secciones["rutas"]),
    )

    df_flota_normalizado = normalizar_flota(
        hoja_completa,
        int(filas_secciones["flota"]),
    )

    df_reposicion_normalizado = normalizar_reposicion(
        hoja_completa,
        int(filas_secciones["reposicion"]),
    )

    df_output_normalizado = normalizar_output_requerido(
        hoja_completa,
        int(filas_secciones["output_requerido"]),
    )

    return {
        "centros": df_centros_normalizado,
        "consumo": df_consumo_normalizado,
        "rutas": df_rutas_normalizado,
        "flota": df_flota_normalizado,
        "reposicion": df_reposicion_normalizado,
        "output_requerido": df_output_normalizado,
    }


datos_modelo = normalizar_datos_grupo3(datos_crudos)

resumen_tablas = pd.DataFrame(
    {
        "tabla": datos_modelo.keys(),
        "filas": [
            len(tabla)
            for tabla in datos_modelo.values()
        ],
        "columnas": [
            len(tabla.columns)
            for tabla in datos_modelo.values()
        ],
    }
)

display(resumen_tablas)

,tabla,filas,columnas
0,centros,4,7
1,consumo,3,7
2,rutas,9,8
3,flota,3,6
4,reposicion,6,8
5,output_requerido,3,3


## Validación de datos

In [11]:
def validar_datos_grupo3(
    datos: dict[str, pd.DataFrame],
) -> None:

    errores = []
    advertencias = []
    pruebas_superadas = []

    tablas_requeridas = {
        "centros",
        "consumo",
        "rutas",
        "flota",
        "reposicion",
        "output_requerido",
    }

    tablas_faltantes = tablas_requeridas.difference(datos.keys())

    if tablas_faltantes:
        errores.append(
            f"Faltan tablas requeridas: {sorted(tablas_faltantes)}"
        )

    if errores:
        raise ValueError("\n".join(errores))

    centros = datos["centros"]
    consumo = datos["consumo"]
    rutas = datos["rutas"]
    flota = datos["flota"]
    reposicion = datos["reposicion"]
    output_requerido = datos["output_requerido"]

    cantidades_esperadas = {
        "centros": (len(centros), 4),
        "suministros": (len(consumo), 3),
        "rutas": (len(rutas), 9),
        "tipos de vehículo": (len(flota), 3),
        "reposiciones": (len(reposicion), 6),
        "outputs requeridos": (len(output_requerido), 3),
    }

    for nombre, (cantidad_real, cantidad_esperada) in (
        cantidades_esperadas.items()
    ):
        if cantidad_real != cantidad_esperada:
            errores.append(
                f"{nombre}: se esperaban {cantidad_esperada} "
                f"registros y se encontraron {cantidad_real}."
            )
        else:
            pruebas_superadas.append(
                f"{nombre}: {cantidad_real} registros"
            )

    columnas_obligatorias = {
        "centros": [
            "centro",
            "zona",
            "agua_litros",
            "alimentos_raciones",
            "medicamentos_kits",
            "capacidad_almacenamiento_m3",
        ],
        "consumo": [
            "suministro",
            "consumo_por_persona_dia",
            "consumo_por_persona_bloque",
        ],
        "rutas": [
            "ruta_id",
            "origen",
            "destino",
            "distancia_km",
            "tiempo_normal_h",
            "capacidad_vehiculos_viaje",
            "estado_ruta",
        ],
        "flota": [
            "tipo_vehiculo",
            "cantidad_disponible",
            "capacidad_m3",
            "consumo_combustible_gal_km",
            "combustible_disponible_gal",
        ],
        "reposicion": [
            "bloque",
            "hora_inicio",
            "hora_fin",
            "agua_adicional_litros",
            "alimentos_adicionales_raciones",
            "medicamentos_adicionales_kits",
            "fuente",
        ],
        "output_requerido": [
            "output_id",
            "nombre_output",
            "descripcion_original",
        ],
    }

    for nombre_tabla, columnas in columnas_obligatorias.items():
        tabla = datos[nombre_tabla]

        faltantes = tabla[columnas].isna().sum()
        faltantes = faltantes[faltantes > 0]

        if not faltantes.empty:
            errores.append(
                f"{nombre_tabla}: valores faltantes en "
                f"{faltantes.to_dict()}."
            )
        else:
            pruebas_superadas.append(
                f"{nombre_tabla}: sin valores faltantes obligatorios"
            )

    identificadores = {
        "centros": "centro",
        "consumo": "suministro",
        "rutas": "ruta_id",
        "flota": "tipo_vehiculo",
        "reposicion": "bloque",
        "output_requerido": "output_id",
    }

    for nombre_tabla, identificador in identificadores.items():
        tabla = datos[nombre_tabla]

        if tabla[identificador].duplicated().any():
            valores_duplicados = tabla.loc[
                tabla[identificador].duplicated(keep=False),
                identificador,
            ].tolist()

            errores.append(
                f"{nombre_tabla}: identificadores duplicados "
                f"{valores_duplicados}."
            )
        else:
            pruebas_superadas.append(
                f"{nombre_tabla}: sin identificadores duplicados"
            )

    columnas_no_negativas = {
        "centros": [
            "agua_litros",
            "alimentos_raciones",
            "medicamentos_kits",
            "capacidad_almacenamiento_m3",
        ],
        "consumo": [
            "consumo_por_persona_dia",
            "consumo_por_persona_bloque",
        ],
        "flota": [
            "cantidad_disponible",
            "capacidad_m3",
            "consumo_combustible_gal_km",
            "combustible_disponible_gal",
        ],
        "reposicion": [
            "agua_adicional_litros",
            "alimentos_adicionales_raciones",
            "medicamentos_adicionales_kits",
        ],
    }

    for nombre_tabla, columnas in columnas_no_negativas.items():
        tabla = datos[nombre_tabla]

        for columna in columnas:
            if (tabla[columna] < 0).any():
                errores.append(
                    f"{nombre_tabla}.{columna} contiene "
                    "valores negativos."
                )

    columnas_positivas_rutas = [
        "distancia_km",
        "tiempo_normal_h",
        "capacidad_vehiculos_viaje",
    ]

    for columna in columnas_positivas_rutas:
        if (rutas[columna] <= 0).any():
            errores.append(
                f"rutas.{columna} contiene valores menores "
                "o iguales a cero."
            )

    bloques_esperados_reposicion = {2, 3, 4, 6, 8, 10}
    bloques_observados = set(reposicion["bloque"])

    if bloques_observados != bloques_esperados_reposicion:
        errores.append(
            "Los bloques de reposición no coinciden con "
            f"{sorted(bloques_esperados_reposicion)}."
        )
    else:
        pruebas_superadas.append(
            "bloques de reposición: T2, T3, T4, T6, T8 y T10"
        )

    duracion_reposiciones = (
        reposicion["hora_fin"] - reposicion["hora_inicio"]
    )

    if not (duracion_reposiciones == PASO_HORAS).all():
        errores.append(
            "Algún intervalo de reposición no dura "
            f"{PASO_HORAS} horas."
        )
    else:
        pruebas_superadas.append(
            "intervalos de reposición: 6 horas"
        )

    if not (
        reposicion["hora_inicio"]
        == reposicion["bloque"] * PASO_HORAS
    ).all():
        errores.append(
            "La hora inicial no coincide con bloque × PASO_HORAS."
        )
    else:
        pruebas_superadas.append(
            "correspondencia correcta entre bloques y horas"
        )

    consumo_esperado_bloque = (
        consumo["consumo_por_persona_dia"]
        * PASO_HORAS
        / 24
    )

    if not np.allclose(
        consumo["consumo_por_persona_bloque"],
        consumo_esperado_bloque,
    ):
        errores.append(
            "Las tasas de consumo por bloque no coinciden con "
            "las tasas diarias convertidas a seis horas."
        )
    else:
        pruebas_superadas.append(
            "tasas por bloque derivadas correctamente"
        )

    estados_permitidos = {
        "Disponible",
        "Dañada — velocidad reducida 60%",
        "Parcialmente bloqueada",
    }

    estados_desconocidos = (
        set(rutas["estado_ruta"]) - estados_permitidos
    )

    if estados_desconocidos:
        errores.append(
            f"Estados de ruta no reconocidos: {estados_desconocidos}."
        )
    else:
        pruebas_superadas.append(
            "estados de ruta originales preservados"
        )

    nombres_centros = set(centros["centro"])
    origenes_rutas = set(rutas["origen"])
    origenes_sin_coincidencia = origenes_rutas - nombres_centros

    if origenes_sin_coincidencia:
        advertencias.append(
            "Algunos nombres de origen no coinciden literalmente "
            "con la tabla de centros: "
            f"{sorted(origenes_sin_coincidencia)}. "
            "Se necesitará un mapeo explícito."
        )

    if errores:
        mensaje = "\n".join(
            f"- {error}" for error in errores
        )
        raise ValueError(
            "La validación encontró errores críticos:\n"
            f"{mensaje}"
        )

    print("VALIDACIÓN GENERAL: PASA")

    print("\nPruebas superadas:")
    for prueba in pruebas_superadas:
        print(f"  PASA — {prueba}")

    if advertencias:
        print("\nAdvertencias que requieren documentación:")
        for advertencia in advertencias:
            print(f"  ADVERTENCIA — {advertencia}")


validar_datos_grupo3(datos_modelo)

VALIDACIÓN GENERAL: PASA

Pruebas superadas:
  PASA — centros: 4 registros
  PASA — suministros: 3 registros
  PASA — rutas: 9 registros
  PASA — tipos de vehículo: 3 registros
  PASA — reposiciones: 6 registros
  PASA — outputs requeridos: 3 registros
  PASA — centros: sin valores faltantes obligatorios
  PASA — consumo: sin valores faltantes obligatorios
  PASA — rutas: sin valores faltantes obligatorios
  PASA — flota: sin valores faltantes obligatorios
  PASA — reposicion: sin valores faltantes obligatorios
  PASA — output_requerido: sin valores faltantes obligatorios
  PASA — centros: sin identificadores duplicados
  PASA — consumo: sin identificadores duplicados
  PASA — rutas: sin identificadores duplicados
  PASA — flota: sin identificadores duplicados
  PASA — reposicion: sin identificadores duplicados
  PASA — output_requerido: sin identificadores duplicados
  PASA — bloques de reposición: T2, T3, T4, T6, T8 y T10
  PASA — intervalos de reposición: 6 horas
  PASA — correspond

### Tabla de trazabilidad de los datos

In [12]:
tabla_trazabilidad = pd.DataFrame(
    [
        {
            "dato": "Inventario inicial de agua",
            "fuente": "Grupo3_CadenaSuministro.xlsx — Centros de acopio",
            "unidad": "litros",
            "uso_en_modelo": (
                "Estado inicial del inventario de agua por centro"
            ),
            "transformacion": (
                "Conversión de formato tabular; valor sin modificar"
            ),
        },
        {
            "dato": "Inventario inicial de alimentos",
            "fuente": "Grupo3_CadenaSuministro.xlsx — Centros de acopio",
            "unidad": "raciones",
            "uso_en_modelo": (
                "Estado inicial del inventario de alimentos por centro"
            ),
            "transformacion": (
                "Conversión de formato tabular; valor sin modificar"
            ),
        },
        {
            "dato": "Inventario inicial de medicamentos",
            "fuente": "Grupo3_CadenaSuministro.xlsx — Centros de acopio",
            "unidad": "kits",
            "uso_en_modelo": (
                "Estado inicial del inventario de medicamentos por centro"
            ),
            "transformacion": (
                "Conversión de formato tabular; valor sin modificar"
            ),
        },
        {
            "dato": "Capacidad de almacenamiento",
            "fuente": "Grupo3_CadenaSuministro.xlsx — Centros de acopio",
            "unidad": "m³",
            "uso_en_modelo": (
                "Referencia de capacidad física de cada centro"
            ),
            "transformacion": (
                "No se aplica todavía como restricción porque faltan "
                "factores de conversión de suministros a m³"
            ),
        },
        {
            "dato": "Consumo per cápita",
            "fuente": "Grupo3_CadenaSuministro.xlsx — Tasas de consumo",
            "unidad": "unidad de suministro/persona/día",
            "uso_en_modelo": (
                "Cálculo de demanda de agua, alimentos y medicamentos"
            ),
            "transformacion": (
                "consumo_diario × PASO_HORAS / 24"
            ),
        },
        {
            "dato": "Distancia de las rutas",
            "fuente": "Grupo3_CadenaSuministro.xlsx — Red de distribución",
            "unidad": "km",
            "uso_en_modelo": (
                "Cálculo de tiempo logístico y consumo de combustible"
            ),
            "transformacion": (
                "Valor sin modificar; falta definir si se considera "
                "viaje de ida o de ida y vuelta"
            ),
        },
        {
            "dato": "Tiempo normal de ruta",
            "fuente": "Grupo3_CadenaSuministro.xlsx — Red de distribución",
            "unidad": "horas",
            "uso_en_modelo": (
                "Programación de llegadas de los vehículos"
            ),
            "transformacion": (
                "Valor sin modificar antes de aplicar el estado de ruta"
            ),
        },
        {
            "dato": "Capacidad de vehículos por viaje",
            "fuente": "Grupo3_CadenaSuministro.xlsx — Red de distribución",
            "unidad": "vehículos/viaje",
            "uso_en_modelo": (
                "Restricción del número de vehículos utilizables por ruta"
            ),
            "transformacion": "Valor sin modificar",
        },
        {
            "dato": "Estado de ruta post-sismo",
            "fuente": "Grupo3_CadenaSuministro.xlsx — Red de distribución",
            "unidad": "categoría",
            "uso_en_modelo": (
                "Modificación de disponibilidad, tiempo o capacidad"
            ),
            "transformacion": (
                "Se conserva el texto original; la interpretación "
                "cuantitativa será un parámetro documentado"
            ),
        },
        {
            "dato": "Cantidad de vehículos",
            "fuente": "Grupo3_CadenaSuministro.xlsx — Flota",
            "unidad": "vehículos",
            "uso_en_modelo": (
                "Restricción de la flota disponible por tipo"
            ),
            "transformacion": "Conversión a número entero",
        },
        {
            "dato": "Capacidad de los vehículos",
            "fuente": "Grupo3_CadenaSuministro.xlsx — Flota",
            "unidad": "m³/vehículo",
            "uso_en_modelo": (
                "Capacidad máxima de transporte por viaje"
            ),
            "transformacion": (
                "No se usa para convertir suministros hasta disponer "
                "de equivalencias de litros, raciones y kits a m³"
            ),
        },
        {
            "dato": "Consumo de combustible",
            "fuente": "Grupo3_CadenaSuministro.xlsx — Flota",
            "unidad": "gal/km",
            "uso_en_modelo": (
                "Cálculo del combustible consumido por viaje"
            ),
            "transformacion": "Valor sin modificar",
        },
        {
            "dato": "Combustible disponible",
            "fuente": "Grupo3_CadenaSuministro.xlsx — Flota",
            "unidad": "galones",
            "uso_en_modelo": (
                "Restricción acumulada de las operaciones logísticas"
            ),
            "transformacion": "Valor sin modificar",
        },
        {
            "dato": "Reposiciones externas",
            "fuente": "Grupo3_CadenaSuministro.xlsx — Reposición",
            "unidad": "litros, raciones y kits",
            "uso_en_modelo": (
                "Flujo de entrada al inventario durante la simulación"
            ),
            "transformacion": (
                "Separación de bloque, hora inicial y hora final"
            ),
        },
        {
            "dato": "Población desplazada",
            "fuente": "Pendiente de output formal del Grupo 1",
            "unidad": "personas por zona y bloque",
            "uso_en_modelo": (
                "Conversión de población en demanda de suministros"
            ),
            "transformacion": (
                "personas × consumo_por_persona_bloque"
            ),
        },
        {
            "dato": "Accesibilidad actualizada",
            "fuente": "Pendiente de output formal del Grupo 4",
            "unidad": "estado, horas e índice entre 0 y 1",
            "uso_en_modelo": (
                "Actualización de rutas, tiempos y restricciones de acceso"
            ),
            "transformacion": (
                "Se definirá después del intercambio presencial"
            ),
        },
    ]
)

display(tabla_trazabilidad)

,dato,fuente,unidad,uso_en_modelo,transformacion
0,Inventario inicial de agua,Grupo3_CadenaSuministro.xlsx — Centros de acopio,litros,Estado inicial del inventario de agua por centro,Conversión de formato tabular; valor sin modif...
1,Inventario inicial de alimentos,Grupo3_CadenaSuministro.xlsx — Centros de acopio,raciones,Estado inicial del inventario de alimentos por...,Conversión de formato tabular; valor sin modif...
2,Inventario inicial de medicamentos,Grupo3_CadenaSuministro.xlsx — Centros de acopio,kits,Estado inicial del inventario de medicamentos ...,Conversión de formato tabular; valor sin modif...
3,Capacidad de almacenamiento,Grupo3_CadenaSuministro.xlsx — Centros de acopio,m³,Referencia de capacidad física de cada centro,No se aplica todavía como restricción porque f...
4,Consumo per cápita,Grupo3_CadenaSuministro.xlsx — Tasas de consumo,unidad de suministro/persona/día,"Cálculo de demanda de agua, alimentos y medica...",consumo_diario × PASO_HORAS / 24
5,Distancia de las rutas,Grupo3_CadenaSuministro.xlsx — Red de distribu...,km,Cálculo de tiempo logístico y consumo de combu...,Valor sin modificar; falta definir si se consi...
6,Tiempo normal de ruta,Grupo3_CadenaSuministro.xlsx — Red de distribu...,horas,Programación de llegadas de los vehículos,Valor sin modificar antes de aplicar el estado...
7,Capacidad de vehículos por viaje,Grupo3_CadenaSuministro.xlsx — Red de distribu...,vehículos/viaje,Restricción del número de vehículos utilizable...,Valor sin modificar
8,Estado de ruta post-sismo,Grupo3_CadenaSuministro.xlsx — Red de distribu...,categoría,"Modificación de disponibilidad, tiempo o capac...",Se conserva el texto original; la interpretaci...
9,Cantidad de vehículos,Grupo3_CadenaSuministro.xlsx — Flota,vehículos,Restricción de la flota disponible por tipo,Conversión a número entero


## Descripción del caso

El modelo representa la cadena de suministro de ayuda humanitaria de Ciudad UVG durante las primeras 72 horas posteriores a un terremoto de magnitud 6.8. El sistema debe distribuir agua potable, raciones de alimentos y kits de medicamentos desde cuatro centros de acopio hacia los albergues afectados, usando una red de nueve rutas y una flota heterogénea de camiones, camionetas y motocicletas.

El horizonte se divide en 12 bloques de 6 horas. En cada bloque, los inventarios pueden aumentar por la llegada de reposiciones externas y disminuir por el consumo asociado con la población desplazada. La distribución está limitada por el estado de las rutas, el número de vehículos que puede circular por cada ruta y el combustible disponible.

La unidad principal de análisis es la combinación de zona, suministro y bloque de tiempo. La heterogeneidad del sistema se conserva mediante inventarios iniciales diferentes entre centros, tasas de consumo específicas para cada suministro, rutas con distintas distancias y condiciones, y vehículos con capacidades y consumos de combustible diferentes.

El modelo debe producir tres resultados principales: el balance de suministros por zona, suministro y bloque; la identificación del primer agotamiento crítico sin intervención adicional; y una recomendación de priorización de rutas y frecuencia de distribución que busque maximizar la cobertura bajo la restricción de combustible disponible.

Durante la fase previa al intercambio se utilizan exclusivamente los datos oficiales del Grupo 3. La demanda por zona y bloque se incorporará posteriormente mediante la proyección formal entregada por el Grupo 1, mientras que las condiciones actualizadas de accesibilidad se incorporarán mediante el output formal del Grupo 4. Ambos escenarios deberán conservarse para identificar si la información recibida cambia o confirma la zona inicialmente considerada más crítica.

## Justificación del paradigma

Para seleccionar el paradigma se consideran la naturaleza de las variables, el nivel de agregación, la importancia del orden temporal y el tipo de pregunta que debe responder el modelo. El sistema combina inventarios que evolucionan continuamente por entradas y salidas con operaciones logísticas que ocurren en momentos específicos. Por ello se adopta una arquitectura híbrida basada en **Dinámica de Sistemas (SD)** y **Simulación de Eventos Discretos (DES)**, implementada sobre bloques de 6 horas.

La Dinámica de Sistemas es apropiada para representar los suministros como *stocks*. El inventario de agua, alimentos y medicamentos aumenta mediante las reposiciones recibidas y disminuye mediante el consumo satisfecho. Esta representación permite analizar las trayectorias de inventario, los déficits acumulados y el momento de agotamiento de cada suministro. Aunque la implementación utiliza ecuaciones en tiempo discreto, conserva la lógica fundamental de stocks y flujos.

La Simulación de Eventos Discretos complementa este componente porque las salidas y llegadas de convoyes, los viajes por las rutas y las reposiciones externas ocurren como eventos identificables. Estos eventos modifican el estado del sistema en momentos determinados y están condicionados por el tiempo de viaje, la capacidad de las rutas, la disponibilidad de vehículos y el combustible restante. Su representación permite distinguir entre inventario almacenado, inventario en tránsito e inventario efectivamente recibido.

La frontera entre ambos componentes se define de la siguiente manera: SD representa el balance agregado de inventarios y demanda por zona, suministro y bloque, mientras que DES representa las operaciones logísticas que trasladan suministros y determinan cuándo una carga pasa a estar disponible en su destino. El estado resultante de los eventos logísticos alimenta las entradas del balance de inventario.

Un modelo exclusivamente de Dinámica de Sistemas sería suficiente para estudiar un inventario agregado si las entregas fueran instantáneas y las rutas no importaran. Sin embargo, representaría con menor precisión los retrasos, bloqueos y llegadas de convoyes que afectan la disponibilidad real de la ayuda. Por otra parte, un modelo exclusivamente DES podría representar cada viaje, pero complicaría innecesariamente el cálculo agregado del consumo y del balance de suministros.

No se selecciona un Modelo Basado en Agentes (ABM) porque los datos disponibles no describen decisiones autónomas, aprendizaje o interacción social entre vehículos, centros o albergues. Los vehículos siguen reglas logísticas centralizadas y no poseen objetivos individuales. Incluir agentes no aportaría capacidad explicativa para justificar el aumento de complejidad.

## Entidades y variables de estado

## Supuestos del modelo

## Ecuaciones y reglas

## Scheduling

## Motor de simulación

## Pruebas de consistencia

## Configuración de incertidumbre

## Ejecución de realizaciones